In [0]:
import mlflow
from mlflow.tracking import MlflowClient

catalog = "workspace"
schema = "financial_sentiment"
model_name = f"{catalog}.{schema}.financial_sentiment_classifier"

mlflow.set_registry_uri("databricks-uc")

client = MlflowClient()

experiment = mlflow.get_experiment_by_name("/Users/jezapataf@eafit.edu.co/financial_sentiment_mlops")

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.macro_f1 DESC"],
    max_results=1
)

best_run_id = runs.iloc[0]["run_id"]
best_macro_f1 = runs.iloc[0]["metrics.macro_f1"]

print("Best run:", best_run_id)
print("Best macro_f1:", best_macro_f1)

if best_macro_f1 >= 0.60:
    model_uri = f"runs:/{best_run_id}/model"

    result = mlflow.register_model(
        model_uri=model_uri,
        name=model_name
    )

    print("Modelo registrado:", result.name)
    print("Versión:", result.version)
else:
    raise Exception(f"Modelo no registrado. macro_f1={best_macro_f1} menor al umbral.")